## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [13]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os

In [14]:

DB_NAME = "vector_db"
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')


if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

MODEL = "openai/gpt-oss-120b"
groq_url = "https://api.groq.com/openai/v1"
# groq = ChatOpenAI(api_key=groq_api_key, base_url=groq_url) 

Groq API Key exists and begins gsk_


In [15]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [16]:
retriever = vectorstore.as_retriever()
# Use Groq for free - OpenAI-compatible API
llm = ChatOpenAI(
    api_key=groq_api_key,
    base_url=groq_url,
    model=MODEL,
    temperature=0
)

In [17]:
retriever.invoke("Who is Avery?")

[Document(id='b9f1a11f-3b8f-4d41-9216-a765274dacac', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [18]:
llm.invoke("Who is Avery?")

AIMessage(content='I’m not sure which “Avery” you’re referring to—there are many people, characters, and even brands with that name. Could you give me a bit more context (e.g., a last name, a field like music, literature, a TV show, etc.) so I can provide the most relevant information?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 75, 'total_tokens': 212, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 62, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.004744334, 'prompt_time': 0.003004656, 'completion_time': 0.291588807, 'total_time': 0.294593463}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'id': 'chatcmpl-7746127a-d2c5-4ffd-9a76-d2dd6a2d1d02', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--3c48ece8-0939-4507-a47b-9b51ddf2

## Time to put this together!

In [19]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [20]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [21]:
answer_question("Who is Averi Lancaster?", [])

'**Avery\u202fLancaster** (sometimes miss‑typed as “Averi”) is one of the founding pillars of Insurellm.\n\n| Detail | Information |\n|--------|--------------|\n| **Full Name** | Avery\u202fLancaster |\n| **Date of Birth** | March\u202f15\u202f,\u202f1985 |\n| **Current Role** | Co‑Founder & Chief Executive Officer (CEO) |\n| **Location** | San\u202fFrancisco, California |\n| **Current Salary** | $225,000 per year |\n| **Company Tenure** | 2015\u202f–\u202fpresent (co‑founded Insurellm) |\n| **Career Highlights** | • Co‑founded Insurellm in 2015 and has steered the company to become a leading Insurance‑Tech provider.<br>• Recognized for innovative leadership and deep expertise in risk management, helping the firm break into the mainstream insurance market.<br>• Prior to Insurellm, served as Senior Product Manager (2013‑2015) at Innovate Insurance Solutions, where she launched groundbreaking insurance products for the tech sector. |\n\nIn short, Avery Lancaster is the visionary CEO who 

In [22]:
gr.ChatInterface(answer_question).launch()

d:\LEARNING\LLM_Engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
